# 2026-09-24 — Επισκευή tickers χωρίς ετήσια δεδομένα
**ΙΣΤΟΡΙΚΟ — δεν ξανατρέχει.** Κρατιέται για να θυμόμαστε τι έγινε και γιατί.

- Εύρημα: XOM, BK, EQR, GPS, FBHS, CDAY είχαν τριμηνιαία αλλά όχι ετήσια.
- Αιτία: αλλαγές ticker/CIK (XOM νέο CIK, BK→BNY 21/5/2026, μετονομασίες).
- Λύση: ingest με το CIK από τη βάση μας, `ON CONFLICT DO NOTHING`.
- Ανοιχτό (για shared-data-layer): CIK fallback στο εβδομαδιαίο ingest, EQR εκτός λίστας SEC.

In [ ]:
# ── SETUP (dev notebooks ζουν στο dev/, αλλά δουλεύουν από τη ρίζα του repo) ──
import os, sys
if os.path.basename(os.getcwd()) == "dev":
    os.chdir("..")                      # ώστε να βρίσκει data/ και quality_scores/
sys.path.insert(0, os.getcwd())

import warnings; warnings.filterwarnings("ignore")
import pandas as pd
import matplotlib.pyplot as plt
from sqlalchemy import text
from db import get_engine
eng = get_engine()
print("Φάκελος εργασίας:", os.getcwd())
from data_quality import data_quality_report
from quality_scores.piotroski import piotroski_f_score, print_report
report = data_quality_report(eng)


In [ ]:
from sqlalchemy import text
universe = pd.read_sql(text(
    "SELECT DISTINCT c.ticker FROM companies c JOIN fundamentals f ON f.ticker = c.ticker"
), eng)["ticker"]
no_annual = sorted(set(universe) - set(report.index))
print(f"{len(no_annual)} εταιρείες χωρίς κανένα ετήσιο 10-K:")
print(no_annual)

92 εταιρείες χωρίς κανένα ετήσιο 10-K:
['ABC', 'ABMD', 'AL', 'ANSS', 'APC', 'ARC', 'ARNC', 'ASC', 'ATVI', 'BHGE', 'BIG', 'BK', 'BLL', 'CDAY', 'CHA', 'CHK', 'CMA', 'CTL', 'CTLT', 'CTRA', 'CTXS', 'DFS', 'DISH', 'DNB', 'DO', 'DRE', 'EA', 'EC', 'EQR', 'FB', 'FBHS', 'FL', 'FLT', 'FRO', 'GPS', 'HBI', 'HCP', 'HES', 'HI', 'HOLX', 'HONA', 'IAC', 'IGT', 'IPG', 'JNPR', 'JWN', 'K', 'LSI', 'MI', 'MMC', 'MRO', 'MTL', 'NCR', 'NLOK', 'NLSN', 'ODP', 'PCH', 'PDCO', 'PEAK', 'PKI', 'POM', 'PX', 'PXD', 'RAD', 'RE', 'RYI', 'SATS', 'SB', 'SE', 'SEE', 'SIVB', 'SNT', 'SNV', 'SOV', 'SRCL', 'SWN', 'TEN', 'TGNA', 'TMK', 'TUP', 'TWTR', 'TWX', 'UCL', 'UK', 'UPC', 'WAI', 'WB', 'WBA', 'WLTW', 'WRK', 'X', 'XOM']


In [ ]:
# Κελί 1 — Επιβεβαίωση της υπόθεσης: έχουν τριμηνιαία αλλά όχι ετήσια;
# Αν ναι, και τα τριμηνιαία είναι πριν την επισκευή του Αυγούστου → επιβεβαιώνεται.
q = text("""
    SELECT f.ticker, ft.name AS form_type, COUNT(*) AS rows_,
           MIN(f.period_end) AS first_period, MAX(f.period_end) AS last_period
    FROM fundamentals f
    JOIN form_type_lookup ft ON ft.id = f.form_type_id
    WHERE f.ticker IN ('XOM', 'BK', 'EQR', 'FB', 'SIVB', 'TWTR')
    GROUP BY f.ticker, ft.name
    ORDER BY f.ticker, ft.name
""")
print(pd.read_sql(q, eng).to_string())

  ticker form_type  rows_ first_period last_period
0     BK      10-Q   1153   2007-12-31  2025-12-31
1    EQR      10-Q   1186   2009-06-30  2026-06-30
2    EQR       8-K     97   2008-12-31  2012-12-31
3     FB      10-Q   1228   2010-12-31  2025-12-31
4   SIVB      10-Q    811   2008-12-31  2021-12-31
5   TWTR      10-Q    739   2012-12-31  2021-12-31
6    XOM    10-K/A     26   2007-12-31  2009-12-31
7    XOM      10-Q   1338   2007-12-31  2026-06-30


In [ ]:
# Κελί 2 — Ομάδα 1: υπάρχουν τα ΝΕΑ tickers με ετήσια δεδομένα;
renamed = {"FB": "META", "ABC": "COR", "BLL": "BALL", "CDAY": "DAY", "FBHS": "FBIN",
           "FLT": "CPAY", "GPS": "GAP", "PEAK": "DOC", "NLOK": "GEN", "PKI": "RVTY",
           "RE": "EG", "WLTW": "WTW", "BHGE": "BKR", "CTL": "LUMN", "TMK": "GL", "CHK": "EXE"}
for old, new in renamed.items():
    status = (f"✅ {report.at[new, 'years_on_file']} χρονιές ({report.at[new, 'first_year']}–{report.at[new, 'last_year']})"
              if new in report.index else "❌ δεν υπάρχει")
    print(f"{old:>5} → {new:<5} {status}")

   FB → META  ✅ 14 χρονιές (2012–2025)
  ABC → COR   ✅ 17 χρονιές (2009–2025)
  BLL → BALL  ✅ 16 χρονιές (2010–2025)
 CDAY → DAY   ❌ δεν υπάρχει
 FBHS → FBIN  ❌ δεν υπάρχει
  FLT → CPAY  ✅ 15 χρονιές (2011–2025)
  GPS → GAP   ❌ δεν υπάρχει
 PEAK → DOC   ✅ 17 χρονιές (2009–2025)
 NLOK → GEN   ✅ 17 χρονιές (2010–2026)
  PKI → RVTY  ✅ 16 χρονιές (2011–2025)
   RE → EG    ✅ 16 χρονιές (2010–2025)
 WLTW → WTW   ✅ 14 χρονιές (2010–2025)
 BHGE → BKR   ✅ 9 χρονιές (2017–2025)
  CTL → LUMN  ✅ 15 χρονιές (2011–2025)
  TMK → GL    ✅ 17 χρονιές (2009–2025)
  CHK → EXE   ✅ 17 χρονιές (2009–2025)


In [ ]:
# ΜΟΝΟ ΑΝΑΓΝΩΣΗ — δεν γράφει στη βάση
from fundamentals import fetch_companyfacts, parse_companyfacts
from ticker_provider import get_cik_for_ticker

for t in ["BK", "EQR", "DAY", "FBIN", "GAP", "XOM"]:
    cik = get_cik_for_ticker(t)
    if cik is None:
        print(f"{t:>5}: ❌ δεν βρέθηκε CIK")
        continue
    df = parse_companyfacts(t, cik, fetch_companyfacts(cik))
    fy = df[(df["metric"] == "Assets") & (df["form_type"] == "10-K") & (df["fiscal_period"] == "FY")]
    years = sorted(fy["period_end"].dt.year.unique())
    print(f"{t:>5}: CIK {cik} | {len(years)} ετήσιες χρονιές: {years}")

   BK: ❌ δεν βρέθηκε CIK
  EQR: ❌ δεν βρέθηκε CIK
  DAY: ❌ δεν βρέθηκε CIK
 FBIN: CIK 1519751 | 15 ετήσιες χρονιές: [np.int32(2011), np.int32(2012), np.int32(2013), np.int32(2014), np.int32(2015), np.int32(2016), np.int32(2017), np.int32(2018), np.int32(2019), np.int32(2020), np.int32(2021), np.int32(2022), np.int32(2023), np.int32(2024), np.int32(2025)]
  GAP: CIK 39911 | 17 ετήσιες χρονιές: [np.int32(2010), np.int32(2011), np.int32(2012), np.int32(2013), np.int32(2014), np.int32(2015), np.int32(2016), np.int32(2017), np.int32(2018), np.int32(2019), np.int32(2020), np.int32(2021), np.int32(2022), np.int32(2023), np.int32(2024), np.int32(2025), np.int32(2026)]
  XOM: CIK 2115436 | 0 ετήσιες χρονιές: []


In [ ]:
# ΜΟΝΟ ΑΝΑΓΝΩΣΗ — δεν γράφει στη βάση
import inspect
from sqlalchemy import text
from fundamentals import fetch_companyfacts, parse_companyfacts, upsert_fundamentals

TARGETS = ["BK", "EQR", "XOM", "GPS", "FBHS", "CDAY"]

# 1. Με ποιο CIK είναι αποθηκευμένα τα δεδομένα τους σήμερα;
q = text("""SELECT ticker, cik, COUNT(*) AS rows_ FROM fundamentals
            WHERE ticker = ANY(:t) GROUP BY ticker, cik ORDER BY ticker, rows_ DESC""")
cik_table = pd.read_sql(q, eng, params={"t": TARGETS})
print(cik_table.to_string(), "\n")

# 2. Τι θα έφερνε το parse με αυτό το CIK;
ciks = cik_table.drop_duplicates("ticker").set_index("ticker")["cik"]
previews = {}
for t, cik in ciks.items():
    df = parse_companyfacts(t, int(cik), fetch_companyfacts(int(cik)))
    previews[t] = df
    fy = df[(df["metric"] == "Assets") & (df["form_type"] == "10-K") & (df["fiscal_period"] == "FY")]
    yrs = sorted(int(y) for y in fy["period_end"].dt.year.unique())
    span = f"{yrs[0]}–{yrs[-1]}" if yrs else "—"
    print(f"{t:>5}: CIK {cik} | {len(yrs)} ετήσιες χρονιές ({span}) | {len(df)} γραμμές συνολικά")

# 3. Η ακριβής υπογραφή της συνάρτησης εγγραφής στην έκδοση που έχεις
print("\nupsert_fundamentals", inspect.signature(upsert_fundamentals))

  ticker      cik  rows_
0     BK  1390777   1153
1   CDAY  1725057    666
2    EQR   906107   1283
3   FBHS  1519751   1409
4    GPS    39911   1534
5    XOM    34088   1338
6    XOM  2115436     26 

   BK: CIK 1390777 | 17 ετήσιες χρονιές (2009–2025) | 1585 γραμμές συνολικά
 CDAY: CIK 1725057 | 7 ετήσιες χρονιές (2018–2024) | 870 γραμμές συνολικά
  EQR: CIK 906107 | 17 ετήσιες χρονιές (2009–2025) | 1670 γραμμές συνολικά
 FBHS: CIK 1519751 | 15 ετήσιες χρονιές (2011–2025) | 1967 γραμμές συνολικά
  GPS: CIK 39911 | 17 ετήσιες χρονιές (2010–2026) | 2132 γραμμές συνολικά
  XOM: CIK 34088 | 17 ετήσιες χρονιές (2009–2025) | 1824 γραμμές συνολικά

upsert_fundamentals (df: pandas.DataFrame, engine: sqlalchemy.engine.base.Engine, caches: dict[str, dict[str, int]] | None = None) -> dict


In [ ]:
# ⛔ ΙΣΤΟΡΙΚΟ ΚΕΛΙ ΕΓΓΡΑΦΗΣ — έχει ήδη τρέξει. Άλλαξε σε True ΜΟΝΟ αν ξέρεις γιατί.
RUN_WRITE = False
assert RUN_WRITE, "Κελί εγγραφής απενεργοποιημένο (δες RUN_WRITE)"

# ── ΕΓΓΡΑΦΗ ─────────────────────────────────────────────────────────────
# Βάλε εδώ ΜΟΝΟ όσες είδες στο Κελί Α με ένα CIK και λογικές χρονιές.
APPROVED = ["BK", "EQR", "XOM", "GPS", "FBHS", "CDAY"]

with eng.connect() as conn:
    WRITE_STARTED_AT = conn.execute(text("SELECT now()")).scalar()
print("Ώρα έναρξης (για αναίρεση):", WRITE_STARTED_AT)

for t in APPROVED:
    r = upsert_fundamentals(previews[t], eng)
    print(f"{t:>5}: {r['inserted']} νέες γραμμές από {r['attempted']} "
          f"(οι υπόλοιπες υπήρχαν ήδη και δεν αγγίχτηκαν)")

Ώρα έναρξης (για αναίρεση): 2026-09-24 07:21:45.438656+00:00
   BK: 432 νέες γραμμές από 1585 (οι υπόλοιπες υπήρχαν ήδη και δεν αγγίχτηκαν)
  EQR: 387 νέες γραμμές από 1670 (οι υπόλοιπες υπήρχαν ήδη και δεν αγγίχτηκαν)
  XOM: 486 νέες γραμμές από 1824 (οι υπόλοιπες υπήρχαν ήδη και δεν αγγίχτηκαν)
  GPS: 598 νέες γραμμές από 2132 (οι υπόλοιπες υπήρχαν ήδη και δεν αγγίχτηκαν)
 FBHS: 558 νέες γραμμές από 1967 (οι υπόλοιπες υπήρχαν ήδη και δεν αγγίχτηκαν)
 CDAY: 204 νέες γραμμές από 870 (οι υπόλοιπες υπήρχαν ήδη και δεν αγγίχτηκαν)


In [ ]:
report = data_quality_report(eng)
for t in APPROVED:
    if t in report.index:
        r = report.loc[t]
        print(f"{t:>5}: ✅ {r['years_on_file']} χρονιές ({r['first_year']}–{r['last_year']}), "
              f"κενά: {r['missing_years'] or 'κανένα'}")
    else:
        print(f"{t:>5}: ❌ ακόμα χωρίς ετήσια")

print_report(piotroski_f_score("XOM", eng))

   BK: ✅ 17 χρονιές (2009–2025), κενά: κανένα
  EQR: ✅ 17 χρονιές (2009–2025), κενά: κανένα
  XOM: ✅ 17 χρονιές (2009–2025), κενά: κανένα
  GPS: ✅ 17 χρονιές (2010–2026), κενά: κανένα
 FBHS: ✅ 15 χρονιές (2011–2025), κενά: κανένα
 CDAY: ✅ 7 χρονιές (2018–2024), κενά: κανένα
── Piotroski F-Score: XOM ──────────────────
   FY2025 vs FY2024  (filed 2026-02-18)
   Score: 4 / 7

   Κερδοφορία
   [PASS]  Κερδοφόρα (ROA > 0)          φέτος 6.4%
   [PASS]  Θετικές ταμειακές ροές       φέτος $51.97B
   [FAIL]  ROA βελτιώθηκε               πέρσι 8.9%  φέτος 6.4%
   [PASS]  Ταμείο > λογιστικά κέρδη     κέρδη $28.84B  ταμείο $51.97B

   Χρέος & ρευστότητα
   [N/A ]  Λιγότερο χρέος               πέρσι —  φέτος —
            ↳ δεν υπάρχει ίδιο μέτρο χρέους και στις δύο χρονιές
   [FAIL]  Καλύτερη ρευστότητα          πέρσι 1.31x  φέτος 1.15x
   [PASS]  Όχι νέες μετοχές             πέρσι 4.35B  φέτος 4.18B
            ↳ μετοχές στο τέλος της χρονιάς

   Αποδοτικότητα
   [N/A ]  Μεικτό περιθώριο ↑     